# LumaPhoto — No-Dataset Neural Network Training

**No accounts. No downloads. No waiting for approval.**

This notebook:
1. Downloads ~5 000 free photos from Wikimedia Commons (no login)
2. Labels them automatically using LumaPhoto's own rule-based analysis
3. Trains an EfficientNet-B0 to predict the 15 enhancement parameters
4. Exports `enhancer_params.onnx` — drop it next to `LumaPhoto.exe`

**Runtime → Change runtime type → T4 GPU → Run all (~90 min)**

In [ ]:
# Cell 1 — GPU check
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else '⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# Cell 2 — Install deps
!pip install -q timm onnx onnxruntime tqdm

In [ ]:
# Cell 3 — Mount Drive (saves the model so you can download it later)
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/LumaPhoto', exist_ok=True)
print('Drive mounted. Model will be saved to /content/drive/MyDrive/LumaPhoto/')

In [ ]:
# Cell 4 — Write the rule-based labeller (port of the C# analysis)
RULES_PY = r"""
import math, numpy as np
from PIL import Image

PARAM_NAMES=["exposure","brilliance","highlights","shadows","contrast",
             "brightness","black_point","saturation","vibrance","warmth",
             "tint","sharpness","definition","noise","vignette"]
NUM_PARAMS=15
EXP,BRI,HIG,SHA,CON,BRT,BPT,SAT,VIB,WRM,TNT,SHP,DEF,NRS,VIG=range(15)
def _c(v,lo,hi): return max(lo,min(hi,v))

def analyze(img):
    a=np.array(img.convert("RGB"),dtype=np.float32)
    h,w=a.shape[:2]
    step=max(1,int(math.sqrt(w*h/40000)))
    f=a[::step,::step].reshape(-1,3)
    r,g,b=f[:,0],f[:,1],f[:,2]
    lum=r*.299+g*.587+b*.114; n=len(f)
    rm,gm,bm,lm=r.mean(),g.mean(),b.mean(),lum.mean()
    ls=float(lum.std())
    co=float(((r-rm)**2+(g-gm)**2+(b-bm)**2).mean()**.5/3)
    dr=float((lum<55).sum()/n); br=float((lum>200).sum()/n)
    hs=float((lum>230).sum()/n)
    rg2=r/np.maximum(g,1)
    sk=((r>100)&(r>g)&(g>b)&(r>b*1.15)&(rg2>=1.05)&(rg2<=1.65)&(lum>60)&(lum<210))
    sk_r=float(sk.sum()/n)
    sky_r=float(((b>r*1.1)&(b>g*1.05)&(lum>100)).sum()/n)
    gr_r=float(((g>r*1.1)&(g>b*1.1)&(lum>40)).sum()/n)
    wm_r=float(((r>180)&(r>g*1.3)&(r>b*1.8)).sum()/n)
    wb=float(rm-bm); cs=float(max(rm,gm,bm)-min(rm,gm,bm))
    if lm<42 and dr>.55 and .004<hs<.18: sc="Night"
    elif ls>68 and dr>.12 and br>.12: sc="HDR"
    elif sk_r>.12: sc="Portrait"
    elif wm_r>.12 and 70<lm<165: sc="Sunset"
    elif sky_r>.15 or gr_r>.20: sc="Landscape"
    elif lm<62 or dr>.45: sc="LowLight"
    elif lm>162 and dr<.05: sc="HighKey"
    elif lm>95 and ls>28: sc="Daylight"
    else: sc="General"
    return dict(scene=sc,l=lm,ls=ls,co=co,dr=dr,br=br,rm=rm,gm=gm,bm=bm,
                sk_r=sk_r,sky_r=sky_r,gr_r=gr_r,wm_r=wm_r,hs=hs,wb=wb,cs=cs)

def compute_params(a):
    p=np.zeros(NUM_PARAMS,dtype=np.float32)
    sc=a["scene"]; l=a["l"]; wb=a["wb"]; cs=a["cs"]
    dr=a["dr"]; br=a["br"]; hs=a["hs"]; co=a["co"]
    if sc=="Night":
        p[EXP]=_c(round((80-l)*.55),0,60);p[BRI]=20;p[HIG]=-38 if hs>.04 else -22
        p[SHA]=45 if dr>.70 else 35;p[CON]=22;p[BRT]=5;p[BPT]=8
        p[VIB]=18;p[WRM]=10 if wb<-15 else 0;p[SHP]=8;p[DEF]=10
        p[NRS]=45 if l<30 else 32;p[VIG]=16
    elif sc=="HDR":
        p[EXP]=_c(round((114-l)*.30),-30,28);p[BRI]=15
        p[HIG]=_c(round(-br*125),-72,-25);p[SHA]=_c(round(dr*88),22,58)
        p[CON]=-5;p[BPT]=12;p[SAT]=8 if cs<20 else 4;p[VIB]=20 if cs<20 else 14
        p[WRM]=_c(-round(wb*.25),-20,-5) if wb>30 else _c(round(-wb*.25),5,20) if wb<-30 else 0
        p[SHP]=10;p[DEF]=20;p[NRS]=15 if l<80 else 5;p[VIG]=10
    elif sc=="Portrait":
        p[EXP]=_c(round((122-l)*.42),-55,50);p[BRI]=20 if l<110 else 14 if l<140 else 8
        p[HIG]=-22 if br>.10 else -5;p[SHA]=32 if dr>.20 else 20 if dr>.10 else 10
        p[CON]=12 if co<30 else 7 if co<50 else 2;p[BRT]=5 if l<80 else 0;p[BPT]=8
        p[SAT]=8 if cs<20 else 4 if cs<40 else 0;p[VIB]=18 if cs<20 else 12 if cs<40 else 6
        p[WRM]=-8 if wb>40 else 15 if wb<-20 else 8 if wb<0 else 3
        p[SHP]=8;p[DEF]=12;p[NRS]=15 if l<70 else 0;p[VIG]=10
    elif sc=="Sunset":
        p[EXP]=_c(round((100-l)*.42),-50,36);p[BRI]=6
        p[HIG]=_c(round(-br*100),-65,-22) if br>.15 else -22;p[SHA]=15
        p[CON]=18 if co<35 else 12;p[BPT]=14;p[SAT]=22;p[VIB]=32
        p[WRM]=0 if wb>20 else 12;p[SHP]=12;p[DEF]=16;p[VIG]=16
    elif sc=="Landscape":
        p[EXP]=_c(round((108-l)*.42),-55,50);p[BRI]=14 if l<110 else 10
        p[HIG]=_c(round(-br*95),-65,-8) if br>.15 else -18 if br>.06 else -5
        p[SHA]=_c(round(dr*72),12,46) if dr>.30 else 15 if dr>.12 else 6
        p[CON]=24 if co<40 else 14 if co<60 else 7;p[BRT]=5 if l<80 else 0;p[BPT]=12
        p[SAT]=18 if cs<20 else 12 if cs<40 else 6;p[VIB]=28 if cs<20 else 20 if cs<40 else 12
        p[WRM]=_c(-round(wb*.28),-28,-5) if wb>35 else _c(round(-wb*.25),5,22) if wb<-25 else 0
        p[SHP]=14;p[DEF]=20;p[VIG]=12
    elif sc=="LowLight":
        p[EXP]=_c(round((95-l)*.42),-20,52);p[BRI]=14
        p[HIG]=-18 if br>.06 else -5;p[SHA]=26 if dr>.50 else 18 if dr>.35 else 12
        p[CON]=22 if co<35 else 14 if co<55 else 6;p[BRT]=5 if l<80 else 0;p[BPT]=8
        p[SAT]=8 if cs<15 else 4;p[VIB]=20 if cs<15 else 14
        p[WRM]=_c(-round(wb*.3),-25,-5) if wb>30 else _c(round(-wb*.3),5,25) if wb<-30 else 0
        p[SHP]=10;p[DEF]=14;p[NRS]=25 if dr>.50 else 20;p[VIG]=8
    elif sc=="HighKey":
        p[EXP]=_c(round((140-l)*.35),-30,18);p[BRI]=6;p[HIG]=_c(round(-br*85),-52,-10)
        p[SHA]=4;p[CON]=14 if co<25 else 6;p[BPT]=5
        p[SAT]=8 if cs<15 else 4;p[VIB]=14 if cs<15 else 8
        p[WRM]=-8 if wb>30 else 8 if wb<-30 else 0;p[SHP]=12;p[DEF]=14;p[VIG]=6
    elif sc=="Daylight":
        p[EXP]=_c(round((112-l)*.30),-25,22);p[BRI]=12 if l<115 else 8
        p[HIG]=-20 if br>.12 else -12 if br>.06 else -4
        p[SHA]=12 if dr>.15 else 5;p[CON]=16 if co<40 else 10 if co<58 else 2;p[BPT]=10
        p[SAT]=10 if cs<20 else 5 if cs<40 else 0;p[VIB]=18 if cs<20 else 12 if cs<40 else 8
        p[WRM]=_c(-round(wb*.25),-22,-5) if wb>30 else _c(round(-wb*.25),5,22) if wb<-30 else 0
        p[SHP]=12;p[DEF]=16;p[VIG]=10
    else:
        p[EXP]=_c(round((112-l)*.42),-55,50);p[BRI]=16 if l<100 else 10 if l<130 else 6
        p[HIG]=_c(round(-br*95),-65,-8) if br>.15 else -16 if br>.06 else -4
        p[SHA]=_c(round(dr*72),12,46) if dr>.30 else 15 if dr>.12 else 6
        p[CON]=_c(round(20+(35-co)*.4),10,32) if co<35 else 10 if co<55 else 2
        p[BRT]=5 if l<80 else 0;p[BPT]=10
        p[SAT]=12 if cs<15 else 6 if cs<35 else 2;p[VIB]=22 if cs<15 else 16 if cs<35 else 10
        p[WRM]=_c(-round(wb*.3),-28,-5) if wb>30 else _c(round(-wb*.3),5,28) if wb<-30 else 0
        p[SHP]=12;p[DEF]=18;p[NRS]=12 if l<70 else 0;p[VIG]=10
    for i in range(11): p[i]=_c(p[i],-100,100)
    for i in range(11,15): p[i]=_c(p[i],0,100)
    return p

def label_image(img): return compute_params(analyze(img))
"""

with open('/content/rules.py','w') as f: f.write(RULES_PY)
print('rules.py written.')

In [ ]:
# Cell 5 — Write model.py
MODEL_PY = """
import torch, torch.nn as nn, timm
NUM_PARAMS=15
PARAM_RANGES=[(-100,100)]*11+[(0,100)]*4
class PhotoEnhancerNet(nn.Module):
    def __init__(self,pretrained=True):
        super().__init__()
        self.backbone=timm.create_model('efficientnet_b0',pretrained=pretrained,num_classes=0,global_pool='avg')
        d=self.backbone.num_features
        self.head=nn.Sequential(nn.Dropout(.35),nn.Linear(d,512),nn.SiLU(),nn.LayerNorm(512),nn.Dropout(.2),nn.Linear(512,256),nn.SiLU(),nn.Linear(256,NUM_PARAMS))
        self._s=[i for i,(lo,_) in enumerate(PARAM_RANGES) if lo<0]
        self._p=[i for i,(lo,_) in enumerate(PARAM_RANGES) if lo==0]
    def forward(self,x):
        r=self.head(self.backbone(x)); o=torch.empty_like(r)
        o[:,self._s]=torch.tanh(r[:,self._s])*100.0
        o[:,self._p]=torch.sigmoid(r[:,self._p])*100.0
        return o
"""
with open('/content/model.py','w') as f: f.write(MODEL_PY)
print('model.py written.')

In [ ]:
# Cell 6 — Download free photos from Wikimedia Commons
import json, urllib.request, urllib.parse, os
from pathlib import Path

WIKI_DIR    = '/content/data/wikimedia'
TARGET      = 5000   # increase for better quality — each 1000 adds ~5 min download
Path(WIKI_DIR).mkdir(parents=True, exist_ok=True)

CATEGORIES = [
    'Landscape photographs','Portrait photographs','Sunset photographs',
    'Night photographs','HDR photographs','Nature photographs',
    'Urban photography','Street photography','Travel photography',
    'Architecture photographs','Flower photographs','Mountain photographs',
    'Forest photographs','Beach photographs','Sky photographs',
    'City photographs','Wildlife photographs','People photographs',
]

def get_image_urls(category, limit=50):
    api = (f'https://commons.wikimedia.org/w/api.php?action=query&format=json'
           f'&list=categorymembers&cmtitle=Category:{urllib.parse.quote(category)}'
           f'&cmtype=file&cmlimit={limit}')
    try:
        with urllib.request.urlopen(api, timeout=15) as r:
            members = json.loads(r.read()).get('query',{}).get('categorymembers',[])
        titles = [m['title'] for m in members
                  if m['title'].lower().endswith(('.jpg','.jpeg','.png'))]
        urls = []
        for title in titles[:limit]:
            iapi = (f'https://commons.wikimedia.org/w/api.php?action=query&format=json'
                    f'&prop=imageinfo&iiprop=url&iiurlwidth=960'
                    f'&titles={urllib.parse.quote(title)}')
            with urllib.request.urlopen(iapi, timeout=15) as r2:
                pages = json.loads(r2.read()).get('query',{}).get('pages',{})
            for pg in pages.values():
                info = pg.get('imageinfo',[])
                if info:
                    urls.append(info[0].get('thumburl') or info[0].get('url',''))
        return [u for u in urls if u]
    except: return []

downloaded = list(Path(WIKI_DIR).glob('*.jpg'))
per_cat = max(10, TARGET // len(CATEGORIES) + 5)

for cat in CATEGORIES:
    if len(downloaded) >= TARGET: break
    urls = get_image_urls(cat, per_cat)
    for url in urls:
        if len(downloaded) >= TARGET: break
        fname = Path(WIKI_DIR) / f'wiki_{len(downloaded):05d}.jpg'
        if not fname.exists():
            try: urllib.request.urlretrieve(url, fname)
            except: continue
        downloaded.append(fname)
    print(f'  {cat[:40]:40s} → {len(downloaded)}/{TARGET}')

print(f'\nDownloaded {len(downloaded)} photos.')
IMAGE_FILES = [str(p) for p in Path(WIKI_DIR).glob('*.jpg')]

In [ ]:
# Cell 7 — Generate labels using the rule-based system
import sys, numpy as np
from PIL import Image
from tqdm import tqdm
sys.path.insert(0,'/content')
from rules import label_image, NUM_PARAMS, PARAM_NAMES

LABEL_CACHE = '/content/labels.npy'

labels = np.zeros((len(IMAGE_FILES), NUM_PARAMS), dtype=np.float32)
bad = 0
for i, path in enumerate(tqdm(IMAGE_FILES, desc='Labelling')):
    try:
        labels[i] = label_image(Image.open(path))
    except:
        bad += 1

np.save(LABEL_CACHE, labels)
print(f'Labels saved. Bad images: {bad}/{len(IMAGE_FILES)}')

# Show distribution
print('\nLabel statistics (mean ± std):')
for i,name in enumerate(PARAM_NAMES):
    print(f'  {name:12s}: {labels[:,i].mean():+6.1f} ± {labels[:,i].std():.1f}')

In [ ]:
# Cell 8 — Dataset + DataLoader
import random, torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class PhotoDataset(Dataset):
    def __init__(self, files, labels, crop=256):
        self.files  = files
        self.labels = labels
        self.aug    = transforms.Compose([
            transforms.RandomCrop(crop),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=.15,contrast=.15,saturation=.1,hue=.03),
        ])
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        try:
            img = Image.open(self.files[i]).convert('RGB')
            w,h = img.size
            if min(w,h) < 288:
                s = 288/min(w,h); img = img.resize((int(w*s),int(h*s)),Image.BICUBIC)
            t = self.aug(transforms.functional.to_tensor(img))
        except:
            t = torch.zeros(3,256,256)
        return t, torch.from_numpy(self.labels[i])

dataset = PhotoDataset(IMAGE_FILES, labels)
loader  = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2,
                     pin_memory=True, drop_last=True)
print(f'{len(dataset)} images, {len(loader)} batches/epoch')

In [ ]:
# Cell 9 — Build model
import torch.nn as nn
from model import PhotoEnhancerNet, NUM_PARAMS

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {DEVICE}')

MEAN = torch.tensor([0.485,0.456,0.406],device=DEVICE).view(1,3,1,1)
STD  = torch.tensor([0.229,0.224,0.225],device=DEVICE).view(1,3,1,1)

model = PhotoEnhancerNet(pretrained=True).to(DEVICE)

optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 3e-5},
    {'params': model.head.parameters(),     'lr': 3e-4},
], weight_decay=1e-4)

EPOCHS = 60
sched  = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=20)
scaler = torch.cuda.amp.GradScaler()

# Parameter weights — more important params get higher loss weight
W = torch.ones(NUM_PARAMS, device=DEVICE)
W[[0,2,4,7]] = 2.0   # exposure, highlights, contrast, saturation
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Cell 10 — Training loop
import math, time, os

os.makedirs('/content/ckpt', exist_ok=True)
best_loss = math.inf

for epoch in range(EPOCHS):
    model.train()
    total = 0.0
    t0    = time.time()

    for img_t, lbl in loader:
        img_t = img_t.to(DEVICE, non_blocking=True)
        lbl   = lbl.to(DEVICE, non_blocking=True)

        small = F.interpolate(img_t, (224,224), mode='bilinear', align_corners=False)
        inp   = (small - MEAN) / STD

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            pred = model(inp)
            loss = ((pred - lbl).pow(2) * W).mean()

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        total += loss.item()

    sched.step()
    avg = total / len(loader)
    print(f'Epoch {epoch+1:02d}/{EPOCHS}  loss={avg:.4f}  ({time.time()-t0:.0f}s)')

    torch.save(model.state_dict(), '/content/ckpt/last.pt')
    if avg < best_loss:
        best_loss = avg
        torch.save(model.state_dict(), '/content/ckpt/best.pt')
        print(f'  ↳ Best saved ({best_loss:.4f})')

print(f'\nDone. Best loss: {best_loss:.4f}')

In [ ]:
# Cell 11 — Export to ONNX + download
import torch.onnx, onnxruntime as ort, shutil

model.eval()
model.load_state_dict(torch.load('/content/ckpt/best.pt', map_location='cpu'))
model = model.cpu()

dummy    = torch.randn(1,3,224,224)
onnx_out = '/content/enhancer_params.onnx'

with torch.no_grad():
    torch.onnx.export(model, dummy, onnx_out, opset_version=17,
        input_names=['input'], output_names=['params'],
        dynamic_axes={'input':{0:'batch'},'params':{0:'batch'}},
        do_constant_folding=True)

# Sanity check
import numpy as np
sess = ort.InferenceSession(onnx_out)
out  = sess.run(None,{'input':np.random.randn(1,3,224,224).astype(np.float32)})[0]
print('Output shape:', out.shape)
for name,val in zip(PARAM_NAMES, out[0]):
    print(f'  {name:12s}: {val:+.1f}')

# Copy to Drive
drive_path = '/content/drive/MyDrive/LumaPhoto/enhancer_params.onnx'
shutil.copy(onnx_out, drive_path)
print(f'\nSaved to Google Drive: {drive_path}')

from google.colab import files
files.download(onnx_out)
print('\nPlace enhancer_params.onnx next to LumaPhoto.exe — done!')